# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing all entities by their `@id`.

Here we print the available record sets, then the fields within each, making sure to always use `@id` values to identify them.

In [ ]:
# List all available record sets and fields by their `@id`
print("Available record sets:")
record_sets = [rs for rs in metadata.record_sets]
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}, name: {getattr(rs, 'name', '<no name>')}")

# List the fields in each record set with their @id
record_set_fields = {}
for rs in record_sets:
    print(f"\nFields for RecordSet @id: {rs.id}")
    fields = [f for f in rs.fields]
    record_set_fields[rs.id] = [field.id for field in fields]
    for f in fields:
        print(f"  Field @id: {f.id}, name: {getattr(f, 'name', '<no name>')}, dataType: {getattr(f, 'data_type', '<unknown>')}")
    if len(fields) == 0:
        print("  (No fields found)")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set and field `@id`s identified in the overview.

*All entities must be referenced by their `@id`.*

In [ ]:
# Extract data from each record set into a pandas DataFrame, using @id everywhere

dataframes = {}

for rs in record_sets:
    record_set_id = rs.id
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")

# For reference in later cells, pick first record set id if any
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f"Defaulting to first record set: {first_record_set_id}")
else:
    first_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. Always use the `@id` to reference fields.

In [ ]:
# Example EDA: select a numeric field and analyze its distribution

# Pick the record set and fields to explore. All references by @id only.
if record_sets:
    rs = record_sets[0]
    rs_id = rs.id
    df = dataframes.get(rs_id)
    numeric_fields = [f for f in rs.fields if getattr(f, 'data_type', None) in ['schema:Integer', 'schema:Number', 'schema:Float']]
    if numeric_fields:
        numeric_field = numeric_fields[0].id
        print(f"Using numeric field with @id: {numeric_field}")
        threshold = df[numeric_field].mean() if numeric_field in df.columns else 0
        if numeric_field in df.columns:
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())
            norm_col = f"{numeric_field}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, norm_col]].head())
        else:
            print(f"Field {numeric_field} not in DataFrame columns: {df.columns.tolist()}")
        # Pick a group field
        group_fields = [f for f in rs.fields if getattr(f, 'data_type', None) == 'schema:Text']
        if group_fields:
            group_field = group_fields[0].id
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field} (mean of numerics):")
                display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: plot the distribution of a numeric field if possible
import matplotlib.pyplot as plt

if record_sets:
    rs = record_sets[0]
    rs_id = rs.id
    df = dataframes.get(rs_id)
    numeric_fields = [f for f in rs.fields if getattr(f, 'data_type', None) in ['schema:Integer', 'schema:Number', 'schema:Float']]
    if numeric_fields and df is not None:
        numeric_field = numeric_fields[0].id
        if numeric_field in df.columns:
            plt.figure(figsize=(8, 4))
            df[numeric_field].hist(bins=20)
            plt.title(f"Distribution of {numeric_field}")
            plt.xlabel(numeric_field)
            plt.ylabel("Frequency")
            plt.show()
        else:
            print(f"Field {numeric_field} not in DataFrame columns: {df.columns.tolist()}")
    else:
        print("No numeric field available for plotting.")
else:
    print("No record sets with data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to use the `mlcroissant` library to load, inspect, and analyze tabular biomedical data described in Croissant schema. All references to schema entities were by `@id`, ensuring reproducibility and clarity. For further analysis, refer to field and record set `@id` values when transforming or visualizing the data.